# Fase 1 — Híbrido (Local / Colab)
**Auto-detecta** si se ejecuta en Google Colab o en PC local.

- En **Colab**: monta Drive y usa rutas de Drive
- En **Local**: usa rutas del sistema de archivos local

**ROI**: Shapefile (`Roigeneral.zip`)
**Bandas**: B2 (Blue), B3 (Green), B4 (Red) — Color natural

In [ ]:
# =============================================================================
# CELDA 1: AUTO-DETECCIÓN DE ENTORNO + INSTALACIÓN
# Funciona en Colab y en PC local
# =============================================================================
import sys, subprocess, os

EN_COLAB = "google.colab" in sys.modules or "google.colab" in str(sys.modules.keys())

if EN_COLAB:
    print("🌐 Entorno detectado: GOOGLE COLAB")
    !pip install pystac-client stackstac rioxarray geopandas rasterio odc-stac -q
else:
    print("💻 Entorno detectado: PC LOCAL")
    # Instalar solo si no están
    try:
        import pystac_client
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install",
            "pystac-client", "stackstac", "rioxarray", "geopandas",
            "rasterio", "odc-stac", "matplotlib", "-q"])

print("✅ Entorno listo.")

In [ ]:
# =============================================================================
# CELDA 2: IMPORTACIONES
# =============================================================================
import os
import glob
import calendar
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import rasterio
import matplotlib
matplotlib.use('Agg')  # Sin interfaz grafica (funciona en ambos entornos)
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from collections import Counter
from pystac_client import Client
import stackstac
from odc.stac import load

if EN_COLAB:
    from google.colab import drive

print("✅ Importaciones completadas.")

In [ ]:
# =============================================================================
# CELDA 3: CONFIGURACIÓN UNIFICADA
# =============================================================================

if EN_COLAB:
    drive.mount('/content/drive')
    SHAPEFILE_PATH = "/content/drive/MyDrive/Tesis/GEE Murcott/ROI/Roigeneral.zip"
    BASE_DIR = os.path.join(os.path.dirname(SHAPEFILE_PATH), "Fase1")
else:
    # Local: el shapefile debe estar en la misma carpeta que este script
    SHAPEFILE_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "Roigeneral.zip")
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = "./Roigeneral.zip"
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = input("📂 Ruta del shapefile: ").strip()
    BASE_DIR = os.path.join(os.path.dirname(os.path.abspath(SHAPEFILE_PATH)), "Fase1")

# ⚙️ PARÁMETROS DE BÚSQUEDA (configura libremente)
MES = 4
ANIO_INICIO = 2025
ANIO_FIN = 2025
# CLOUD_FILTER = {"eo:cloud_cover": {"lt": 10}}  # Descomentar para filtrar
CLOUD_FILTER = None

# ─── Calcular fechas ───────────────────────────────────────────────────────
MES_NOMBRE = ["", "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
               "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
FECHAS = []
for anio in range(ANIO_INICIO, ANIO_FIN + 1):
    ultimo_dia = calendar.monthrange(anio, MES)[1]
    desde = f"{anio}-{MES:02d}-01"
    hasta = f"{anio}-{MES:02d}-{ultimo_dia}"
    FECHAS.append((desde, hasta, f"{MES_NOMBRE[MES]} {anio}"))

print("✅ Configuración cargada.")
print(f"   Shapefile: {SHAPEFILE_PATH}")
print(f"   Período: {FECHAS[0][2] if FECHAS else 'N/A'}")
print(f"   Exportar a: {BASE_DIR}")

In [ ]:
# =============================================================================
# CELDA 4: FUNCIONES AUXILIARES
# =============================================================================

def tif_to_png_batch(input_dir, output_dir, cmap_name='viridis', percentiles=(2, 98)):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 Carpeta creada: {output_dir}")
    files = sorted(glob.glob(os.path.join(input_dir, "*.tif")))
    print(f"🚀 Procesando {len(files)} archivos...")
    for tif_path in files:
        filename = os.path.basename(tif_path).replace('.tif', '.png')
        save_path = os.path.join(output_dir, filename)
        with rasterio.open(tif_path) as src:
            data = src.read(1).astype(np.float32)
            nodata = src.nodata
            if nodata is not None:
                data[data == nodata] = np.nan
        vmin, vmax = np.nanpercentile(data, percentiles)
        norm = Normalize(vmin=vmin, vmax=vmax, clip=True)
        plt.figure(figsize=(10, 10))
        plt.imshow(data, cmap=cmap_name, norm=norm)
        plt.axis('off')
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
        plt.close()
        print(f"✅ Convertido: {filename} [Paleta: {cmap_name}]")

print("✅ Funciones auxiliares cargadas.")

In [ ]:
# =============================================================================
# CELDA 5: CARGAR SHAPEFILE + CALCULAR BBOX
# =============================================================================

print("🔍 Cargando shapefile...")
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"✅ Shapefile cargado: {len(gdf)} feature(s)")

if gdf.crs and gdf.crs.is_geographic:
    gdf_geo = gdf
else:
    gdf_geo = gdf.to_crs("EPSG:4326")

bbox = gdf_geo.total_bounds
print(f"   Bounding Box: {bbox}")

# Visualizar
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
gdf_geo.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=1.5)
ax.set_title("ROI — Mandarina Murcott", fontsize=12)
plt.tight_layout()
plt.show()

# Crear directorios de salida
os.makedirs(f"{BASE_DIR}/PNG", exist_ok=True)
os.makedirs(f"{BASE_DIR}/GeoTIFF", exist_ok=True)
print(f"✅ Directorios creados en: {BASE_DIR}")

In [ ]:
# =============================================================================
# CELDA 6: BÚSQUEDA STAC + CONTEO + CLASIFICACIÓN DE NUBES
# =============================================================================

print("🔍 Conectando al catálogo Earth Search (AWS)...")
catalog = Client.open("https://earth-search.aws.element84.com/v1")
print("✅ Conexión exitosa.")

resultados_totales = []
for fecha_inicio, fecha_fin, label in FECHAS:
    print(f"\n📅 {label} ({fecha_inicio} → {fecha_fin}):")
    params = dict(
        collections=["sentinel-2-l2a"],
        bbox=list(bbox),
        datetime=f"{fecha_inicio}/{fecha_fin}",
    )
    if CLOUD_FILTER:
        params["query"] = CLOUD_FILTER
    search = catalog.search(**params)
    items = list(search.items())
    print(f"   Imágenes encontradas: {len(items)}")

    for item in items:
        cloud = item.properties.get("eo:cloud_cover", None)
        if cloud is None:
            estado = "Sin dato"; cloud_val = -1
        elif cloud < 20:
            estado = "Despejada"; cloud_val = cloud
        elif cloud < 60:
            estado = "Parcial"; cloud_val = cloud
        else:
            estado = "Nublada"; cloud_val = cloud
        resultados_totales.append({
            "fecha": item.datetime.strftime("%Y-%m-%d"),
            "id": item.id, "nubes_%": cloud_val,
            "estado": estado, "mes": label,
        })

df_resultados = pd.DataFrame(resultados_totales)
if len(df_resultados) == 0:
    print("\n❌ No se encontraron imágenes.")
else:
    print("\n" + "=" * 50)
    for (mes, fecha), group in df_resultados.groupby(["mes", "fecha"]):
        nubes_prom = group["nubes_%"].mean()
        if nubes_prom < 0:
            icon = "❓"
            estado_txt = "Sin dato"
        elif nubes_prom < 20:
            icon = "🌤️"
            estado_txt = f"Despejada ({nubes_prom:.0f}%)"
        elif nubes_prom < 60:
            icon = "⛅"
            estado_txt = f"Parcial ({nubes_prom:.0f}%)"
        else:
            icon = "☁️"
            estado_txt = f"Nublada ({nubes_prom:.0f}%)"
        print(f"  {fecha}: {len(group)} img | {icon} {estado_txt}")
    print(f"\n📈 TOTAL: {len(df_resultados)} escenas.")

In [ ]:
# =============================================================================
# CELDA 7: CARGA DEL DATA CUBE
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay imágenes para cargar.")
else:
    print("📦 Cargando Data Cube...")
    items_list = []
    for fecha_inicio, fecha_fin, label in FECHAS:
        params = dict(
            collections=["sentinel-2-l2a"],
            bbox=list(bbox),
            datetime=f"{fecha_inicio}/{fecha_fin}",
        )
        if CLOUD_FILTER:
            params["query"] = CLOUD_FILTER
        search = catalog.search(**params)
        items_list.extend(list(search.items()))
    print(f"   Total escenas: {len(items_list)}")

    ds = load(
        items_list,
        bands=["blue", "green", "red"],
        bbox=list(bbox),
        crs="EPSG:4326",
        resolution=0.0001,
        groupby=None,
        chunks={'time': 1, 'x': 512, 'y': 512},
    )
    print(f"✅ Data Cube: {dict(ds.sizes)}")

In [ ]:
# =============================================================================
# CELDA 8: VISUALIZACIÓN RGB
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos.")
else:
    print("🖼️ Generando visualización RGB...")
    red_band = ds["red"].values / 10000.0
    green_band = ds["green"].values / 10000.0
    blue_band = ds["blue"].values / 10000.0

    n_times = ds.sizes["time"]
    n_cols = min(4, n_times)
    n_rows = max(1, (n_times + n_cols - 1) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = [axes] if n_times == 1 else axes.flatten()

    for t in range(n_times):
        ax = axes[t]
        rgb = np.stack([red_band[t], green_band[t], blue_band[t]], axis=-1)
        rgb = np.clip(rgb, 0, 1)
        ax.imshow(rgb)
        ax.axis('off')
        ax.set_title(str(ds.time.values[t])[:10], fontsize=9)

    for t in range(n_times, len(axes)):
        axes[t].axis('off')
    plt.suptitle("RGB — Mandarina Murcott", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/PNG/Grid_RGB.png", dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✅ Grid guardado.")

In [ ]:
# =============================================================================
# CELDA 9: EXPORTAR GeoTIFF
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos para exportar.")
else:
    print("💾 Exportando GeoTIFFs...")
    for t in range(ds.sizes["time"]):
        fecha = str(ds.time.values[t])[:10]
        geotiff_path = f"{BASE_DIR}/GeoTIFF/Mandarina_{fecha}_RGB.tif"
        rgb_da = ds.isel(time=t)[["red", "green", "blue"]].to_array(dim="band")
        rgb_da.rio.to_raster(geotiff_path)
        print(f"   ✅ GeoTIFF: Mandarina_{fecha}_RGB.tif")
    print(f"📁 GeoTIFFs en: {BASE_DIR}/GeoTIFF/")

In [ ]:
# =============================================================================
# CELDA 10: EXPORTAR PNG (optimizado)
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos para exportar.")
else:
    print("🖼️ Exportando PNGs...")
    for t in range(ds.sizes["time"]):
        fecha = str(ds.time.values[t])[:10]
        png_path = f"{BASE_DIR}/PNG/Mandarina_{fecha}_RGB.png"
        escena = ds.isel(time=t)
        rgb = np.stack([
            escena["red"].values / 10000.0,
            escena["green"].values / 10000.0,
            escena["blue"].values / 10000.0,
        ], axis=-1)
        rgb = np.clip(rgb, 0, 1)
        plt.figure(figsize=(4, 4))
        plt.imshow(rgb, interpolation='nearest')
        plt.title(f"Mandarina — {fecha}", fontsize=9)
        plt.axis('off')
        plt.savefig(png_path, bbox_inches='tight', pad_inches=0, dpi=150)
        plt.close()
        print(f"   ✅ PNG: Mandarina_{fecha}_RGB.png")
    print(f"📁 PNGs en: {BASE_DIR}/PNG/")

In [ ]:
# =============================================================================
# CELDA 11: REPORTE TXT
# =============================================================================

if len(df_resultados) == 0:
    print("❌ No hay datos para reportar.")
else:
    ruta_reporte = f"{BASE_DIR}/Reporte_Fase1.txt"
    with open(ruta_reporte, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write(f"   FASE 1 — {MES_NOMBRE[MES]} {ANIO_INICIO}-{ANIO_FIN}\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"  Bounding Box: {bbox}\n\n")
        for label in df_resultados["mes"].unique():
            f.write(f"  {label}:\n")
            subset = df_resultados[df_resultados["mes"] == label]
            for fecha, grupo in subset.groupby("fecha"):
                nubes_prom = grupo["nubes_%"].mean()
                icon = "Despejada" if nubes_prom < 20 else "Parcial" if nubes_prom < 60 else "Nublada" if nubes_prom >= 0 else "Sin dato"
                f.write(f"    - {fecha}: {len(grupo)} img [{icon}]\n")
            f.write(f"  Subtotal: {len(subset)} imágenes\n\n")
        f.write(f"  TOTAL: {len(df_resultados)} imágenes\n")
        f.write("=", 60)
    print(f"✅ Reporte: {ruta_reporte}")
    print("=" * 60)
    print("   🎉 FASE 1 COMPLETADA")
    print("=" * 60)